In [0]:
# -------------------------------
# Imports
# -------------------------------
import traceback
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

from pyspark.sql import DataFrame
from pyspark.sql import functions as sf
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
    FloatType, TimestampType, BooleanType
)

from pyspark.sql.utils import AnalysisException
from delta.tables import DeltaTable

In [0]:

# -------------------------------
# Global Log
# -------------------------------
log_collection = []


In [0]:
def is_table_exist(layer, schema, table):
    table_name  = f'{layer}.{schema}.{table}'
    try:
        if spark.catalog.tableExists(table_name):
            print(f"✅ Table {table_name} exists.")
            return True
        else:
            print(f"❌ Table {table_name} does not exist.")
            return False
    except AnalysisException as e:
        print(f"Error checking table: {e}")

In [0]:
# -------------------------------
# Utility Functions
# -------------------------------
def select_columns(columns, df):
    return df.select(*columns)

def add_hash_col(columns, df):
    return df.withColumn(
        "row_check_sum",
        sf.md5(sf.concat_ws("|", *[sf.col(c) for c in columns]))
    )

def generate_primary_key(df, primary_key):
    if primary_key and "|" in primary_key:
        cols = primary_key.split("|")
        df = df.withColumn("primary_key", sf.md5(sf.concat(*[sf.col(c) for c in cols])))
        return "primary_key", df
    return primary_key, df

def add_scd_columns(df, incremental_column):
    if isinstance(incremental_column, str):
        effective_from = sf.col(incremental_column)
    else:
        effective_from = sf.current_timestamp()

    return (
        df
        .withColumn("__effective_from_date", effective_from.cast(TimestampType()))
        .withColumn("__effective_to_date", sf.lit("9999-12-01").cast(TimestampType()))
        .withColumn("__is_active", sf.lit(1))
    )

In [0]:

# -------------------------------
# INITIAL LOAD
# -------------------------------
def scd_initial_full_load(
    raw_df, catalog, schema, table, primary_key, incremental_column, selected_columns
):

    df = select_columns(selected_columns, raw_df)
    if primary_key:
        primary_key, df = generate_primary_key(df, primary_key)


    if primary_key and incremental_column:
        window = Window.partitionBy(primary_key).orderBy(sf.col(incremental_column).desc())
        df = (
            df.withColumn("rn", sf.row_number().over(window))
              .filter("rn = 1")
              .drop("rn")
        )
    
    
    df = add_scd_columns(df, incremental_column)

    fully_qualified_table_name = f"{catalog}.{schema}.{table}"
    write_table(df,fully_qualified_table_name, "overwrite")

In [0]:
# -------------------------------
# SCD TYPE 2
# -------------------------------
def scd_type_2(df_source, df_sink, primary_key, incremental_column,catalog,  schema, table):

    df_sink_active = df_sink.filter("__is_active = 1")
    df_sink_inactive = df_sink.filter("__is_active = 0")

    # New records
    df_new = df_source.join(df_sink_active, primary_key, "leftanti")
    df_new = add_scd_columns(df_new, incremental_column)
    
    # Updated
    df_updated = df_source.alias("s").join(
        df_sink_active.alias("t"),
        (sf.col(f"s.{primary_key}") == sf.col(f"t.{primary_key}")) &
        (sf.col("s.row_check_sum") != sf.col("t.row_check_sum"))
    ).select("s.*")


    df_updated = add_scd_columns(df_updated, incremental_column)
    # Expire old
    cols = [c for c in df_sink_active.columns if c != "__effective_to_date" and c != "__is_active" ]

    df_expired = (
        df_sink_active.alias("t")
        .join(df_updated.alias("s"), primary_key)
        .select(
            *[sf.col(f"t.{c}") for c in cols],
            sf.lit(0).alias("__is_active"),
            (
                sf.col(f"s.{incremental_column}") -
                sf.expr("INTERVAL 1 MINUTE")
            ).alias("__effective_to_date")
        )
    )

    df_remaining = df_sink_active.join(df_expired, primary_key, "leftanti")
    # df_new = add_scd_columns(df_new, incremental_column)
    df_final = df_sink_inactive.unionByName(df_expired, allowMissingColumns=True).unionByName(df_new, allowMissingColumns=True).unionByName(df_updated, allowMissingColumns=True).unionByName(df_remaining, allowMissingColumns=True)

    cols = [c for c in df_sink_active.columns if c != "rn" and c != "row_check_sum" ]
    df_final = df_final.select(*cols)

    fully_qualified_table_name = f"{catalog}.{schema}.{table}"
    write_table(df_final,fully_qualified_table_name, "overwrite")

In [0]:
# -------------------------------
# MAIN SCD ORCHESTRATOR
# -------------------------------
def scd_process(raw_df, catalog, schema, table, primary_key, incremental_column, selected_columns):

    if not is_table_exist(catalog, schema, table):
        scd_initial_full_load(raw_df, catalog, schema, table, primary_key, incremental_column, selected_columns)
        return
    sink_table_name = f"{catalog}.{schema}.{table}"
    df_sink = read_table(sink_table_name)
    df_source = select_columns(selected_columns, raw_df)
    df_source = add_hash_col(selected_columns, df_source)
    df_sink = add_hash_col(selected_columns, df_sink)
    primary_key, df_source = generate_primary_key(df_source, primary_key)
    primary_key, df_sink = generate_primary_key(df_sink, primary_key)
    if df_source is not None:
        print("performing SCD.....")
        scd_type_2(df_source, df_sink, primary_key, incremental_column,catalog,  schema, table)

In [0]:

# -------------------------------
# ENTRY FUNCTION
# -------------------------------

def process_scd_table(row, raw_df, selected_columns, incremental_column, primary_key):
    catalog = row['sink_catalog']
    schema = row['sink_schema']
    table = row['sink_table_name']

    try:
        if raw_df is not None:
            scd_process(raw_df, catalog, schema, table, primary_key, incremental_column, selected_columns)
        else:
            log_collection.append([
                "SCD", schema, table, "Skipped", "Empty source", datetime.now()
            ])

    except Exception as e:
        log_collection.append([
            "SCD", schema, table, "Fail",
            f"{str(e)}\n{traceback.format_exc()}",
            datetime.now()
        ])